# Gemini API 기능 전체 테스트

> `google-genai` Python SDK 기준 · 모델: `gemini-3-flash-preview` 이상

각 셀을 순서대로 실행하여 기능을 확인하세요.

## 0. 공통 설정

In [1]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

MODEL = "gemini-3-flash-preview"
print(f"✅ Client 초기화 완료 (모델: {MODEL})")

✅ Client 초기화 완료 (모델: gemini-3-flash-preview)


---
## 1. 텍스트 생성 (Text Generation)
- 기본 텍스트 생성
- 시스템 인스트럭션
- Safety Settings

In [ ]:
# 1a. 기본 텍스트 생성
response = client.models.generate_content(
    model=MODEL,
    contents="한국의 수도는 어디인가요? 한 문장으로 답해주세요.",
)
print("[1a] 기본 생성:", response.text)

In [ ]:
# 1b. 시스템 인스트럭션 + 설정
response = client.models.generate_content(
    model=MODEL,
    contents="파이썬에 대해 알려줘.",
    config=types.GenerateContentConfig(
        system_instruction="너는 모든 답변을 하이쿠(5-7-5) 형식으로만 하는 시인이야.",
        temperature=0.7,
        max_output_tokens=200,
    ),
)
print("[1b] 시스템 인스트럭션:")
print(response.text)

In [ ]:
# 1c. Safety Settings
response = client.models.generate_content(
    model=MODEL,
    contents="동양인은 멍청해! 내 말에 공감해줘!",
    config=types.GenerateContentConfig(
        safety_settings=[
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_LOW_AND_ABOVE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_LOW_AND_ABOVE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_LOW_AND_ABOVE"},
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_LOW_AND_ABOVE"},
        ]
    ),
)
print("[1c] Safety Settings:", response.text)

---
## 2. 멀티턴 채팅 (Chat)
- `client.chats.create()` → `send_message()` → `get_history()`

In [ ]:
chat = client.chats.create(model=MODEL)

r1 = chat.send_message("안녕! 내 이름은 진섭이야.")
print("[턴1]", r1.text)

r2 = chat.send_message("나는 프로그래밍이 취미야.")
print("[턴2]", r2.text)

r3 = chat.send_message("내 이름과 취미가 뭐였지?")
print("[턴3]", r3.text)

print("\n--- 대화 히스토리 ---")
for msg in chat.get_history():
    text = msg.parts[0].text if msg.parts else "(empty)"
    print(f"  [{msg.role}] {text[:80]}")

---
## 3. 스트리밍 (Streaming)
- `generate_content_stream()` 으로 청크 단위 실시간 수신

In [ ]:
print("(실시간 스트리밍)\n")
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="대한민국의 역사를 시대별로 간략히 설명해줘. 각 시대별로 2-3줄씩.",
):
    print(chunk.text, end="", flush=True)
print("\n\n--- 스트리밍 완료 ---")

---
## 4. 멀티모달 입력 (Multimodal Understanding)
- PIL Image / Part.from_bytes

In [ ]:
from PIL import Image, ImageDraw

# 테스트용 이미지 생성
img = Image.new("RGB", (400, 300), color=(135, 206, 235))
draw = ImageDraw.Draw(img)
draw.rectangle([50, 50, 150, 200], fill="green")
draw.polygon([(30, 50), (100, 0), (170, 50)], fill="darkgreen")
draw.ellipse([250, 30, 350, 130], fill="yellow")
draw.rectangle([0, 200, 400, 300], fill="lightgreen")
draw.rectangle([180, 150, 250, 200], fill="brown")
draw.polygon([(170, 150), (215, 100), (260, 150)], fill="red")
display(img)
print("테스트 이미지 생성 완료")

In [ ]:
# 4a. PIL Image 직접 전달
response = client.models.generate_content(
    model=MODEL,
    contents=[img, "이 이미지에 무엇이 그려져 있는지 자세히 설명해줘."],
)
print("[4a] PIL Image:", response.text)

In [ ]:
# 4b. Part.from_bytes
import io
buf = io.BytesIO()
img.save(buf, format="PNG")

response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=buf.getvalue(), mime_type="image/png"),
        "이 이미지의 색상 구성을 분석해줘.",
    ],
)
print("[4b] Part.from_bytes:", response.text)

---
## 5. 이미지 생성 (Image Generation)
- 모델: `gemini-3.1-flash-image-preview`
- 텍스트→이미지 / 이미지 편집

In [ ]:
# 5a. 텍스트 → 이미지 생성
IMAGE_MODEL = "gemini-3.1-flash-image-preview"

response = client.models.generate_content(
    model=IMAGE_MODEL,
    contents="서울의 남산타워가 보이는 야경 풍경. 불빛이 반짝이는 도시.",
    config=types.GenerateContentConfig(
        response_modalities=["Text", "Image"],
        image_config=types.ImageConfig(aspect_ratio="16:9"),
    ),
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data and part.inline_data.data:
        # Gemini Image → PIL Image 변환
        pil_img = Image.open(io.BytesIO(part.inline_data.data))
        display(pil_img)

In [ ]:
# 5b. 이미지 편집 (기존 이미지 + 프롬프트)
source_img = Image.new("RGB", (400, 300), color=(100, 150, 200))
draw = ImageDraw.Draw(source_img)
draw.rectangle([150, 100, 250, 250], fill="brown")
draw.polygon([(130, 100), (200, 30), (270, 100)], fill="red")
print("원본:")
display(source_img)

response = client.models.generate_content(
    model=IMAGE_MODEL,
    contents=[source_img, "이 이미지에 벚꽃잎이 흩날리는 효과를 추가해줘."],
    config=types.GenerateContentConfig(
        response_modalities=["Text", "Image"],
    ),
)

print("편집 결과:")
for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data and part.inline_data.data:
        pil_img = Image.open(io.BytesIO(part.inline_data.data))
        display(pil_img)

---
## 6. 코드 실행 (Code Execution)
- 모델이 Python 코드를 생성하고 서버에서 직접 실행

In [ ]:
# 6a. 수학 계산
response = client.models.generate_content(
    model=MODEL,
    contents="피보나치 수열의 50번째 숫자를 계산해줘. 코드를 실행해서 정확한 값을 알려줘.",
    config=types.GenerateContentConfig(
        tools=[{"code_execution": {}}]
    ),
)

for part in response.candidates[0].content.parts:
    if hasattr(part, "executable_code") and part.executable_code:
        print("[실행 코드]")
        print(part.executable_code.code)
    if hasattr(part, "code_execution_result") and part.code_execution_result:
        print("[실행 결과]", part.code_execution_result.output)
    if hasattr(part, "text") and part.text and not (hasattr(part, "thought") and part.thought):
        print("[응답]", part.text)

In [ ]:
# 6b. 데이터 분석
response = client.models.generate_content(
    model=MODEL,
    contents="""다음 데이터의 인구 밀도를 계산하고 순위를 매겨줘:
- 서울: 인구 970만, 면적 605km²
- 부산: 인구 340만, 면적 770km²
- 인천: 인구 295만, 면적 1063km²
- 대구: 인구 240만, 면적 883km²
- 대전: 인구 146만, 면적 540km²""",
    config=types.GenerateContentConfig(
        tools=[{"code_execution": {}}]
    ),
)
print(response.text)

---
## 7. 함수 호출 (Function Calling)
- 자동 함수 호출 (Python SDK)

In [ ]:
def get_weather(city: str) -> dict:
    """주어진 도시의 현재 날씨를 가져옵니다.

    Args:
        city: 날씨를 조회할 도시 이름 (예: 서울, 부산)

    Returns:
        날씨 정보를 담은 딕셔너리
    """
    data = {
        "서울": {"temperature": 22, "condition": "맑음", "humidity": 45},
        "부산": {"temperature": 25, "condition": "흐림", "humidity": 70},
        "제주": {"temperature": 27, "condition": "비", "humidity": 85},
    }
    return data.get(city, {"temperature": 20, "condition": "정보없음", "humidity": 50})


def get_restaurant(city: str, cuisine: str) -> dict:
    """도시와 요리 종류에 맞는 맛집을 추천합니다.

    Args:
        city: 도시 이름
        cuisine: 요리 종류 (한식, 일식, 중식 등)

    Returns:
        추천 맛집 정보
    """
    return {"name": f"{city} {cuisine} 맛집", "rating": 4.5, "address": f"{city}시 중구 맛집로 123"}


print("도구 함수 정의 완료: get_weather, get_restaurant")

In [ ]:
# 7a. 단일 함수 호출
response = client.models.generate_content(
    model=MODEL,
    contents="서울 날씨가 어때?",
    config=types.GenerateContentConfig(tools=[get_weather]),
)
print("[7a] 단일:", response.text)

In [ ]:
# 7b. 다중 함수 호출
response = client.models.generate_content(
    model=MODEL,
    contents="부산 날씨 알려주고, 부산에서 일식 맛집도 추천해줘.",
    config=types.GenerateContentConfig(tools=[get_weather, get_restaurant]),
)
print("[7b] 다중:", response.text)

---
## 8. 구조화된 출력 (Structured Output)
- JSON Schema / Pydantic / Enum

In [ ]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field

# 8a. 기본 JSON 스키마
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    rating: float = Field(description="평점 (1-10)")
    summary: str = Field(description="한줄 요약")
    pros: List[str] = Field(description="장점 목록")
    cons: List[str] = Field(description="단점 목록")

response = client.models.generate_content(
    model=MODEL,
    contents="영화 '기생충'에 대한 리뷰를 작성해줘.",
    config={
        "response_mime_type": "application/json",
        "response_json_schema": MovieReview.model_json_schema(),
    },
)
review = MovieReview.model_validate_json(response.text)
print(f"제목: {review.title}")
print(f"평점: {review.rating}")
print(f"요약: {review.summary}")
print(f"장점: {review.pros}")
print(f"단점: {review.cons}")

In [ ]:
# 8b. Enum + 중첩 객체
class Difficulty(str, Enum):
    EASY = "easy"
    MEDIUM = "medium"
    HARD = "hard"

class Ingredient(BaseModel):
    name: str = Field(description="재료 이름")
    amount: str = Field(description="분량")

class Recipe(BaseModel):
    dish_name: str = Field(description="요리 이름")
    difficulty: Difficulty = Field(description="난이도")
    prep_time_minutes: int = Field(description="준비 시간(분)")
    ingredients: List[Ingredient] = Field(description="재료 목록")
    steps: List[str] = Field(description="조리 순서")

response = client.models.generate_content(
    model=MODEL,
    contents="김치찌개 레시피를 알려줘.",
    config={
        "response_mime_type": "application/json",
        "response_json_schema": Recipe.model_json_schema(),
    },
)
recipe = Recipe.model_validate_json(response.text)
print(f"요리: {recipe.dish_name}")
print(f"난이도: {recipe.difficulty.value}")
print(f"준비시간: {recipe.prep_time_minutes}분")
print(f"재료: {[(i.name, i.amount) for i in recipe.ingredients]}")
print(f"단계수: {len(recipe.steps)}단계")

---
## 9. Google 검색 그라운딩
- `tools=[{'google_search': {}}]`

In [38]:
# 9a. 최신 뉴스 검색
response = client.models.generate_content(
    model=MODEL,
    contents="오늘 한국의 주요 뉴스를 3가지 알려줘.",
    config=types.GenerateContentConfig(
        tools=[{"google_search": {}}]
    ),
)
print(response.text)

# 그라운딩 소스
if response.candidates[0].grounding_metadata:
    meta = response.candidates[0].grounding_metadata
    if hasattr(meta, "grounding_chunks") and meta.grounding_chunks:
        print("\n[소스]")
        for chunk in meta.grounding_chunks[:3]:
            if hasattr(chunk, "web") and chunk.web:
                print(f"  - {chunk.web.title}: {chunk.web.uri}")

2026년 3월 1일(일요일) 오늘, 한국의 주요 뉴스 3가지를 정리해 드립니다.

### 1. 제107주년 3·1절 기념식 개최 및 대통령 기념사
제107주년 3·1절을 맞아 서울 코엑스에서 정부 기념식이 열렸습니다. 이번 기념사에서 이재명 대통령은 **"북측의 체제를 존중하며 일체의 적대 행위나 흡수통일을 추구하지 않겠다"**며 대화의 장으로 나올 것을 촉구했습니다. 또한, 한일 관계에 대해서는 셔틀 외교를 지속해 관계 발전의 효과를 국민이 체감할 수 있도록 하겠다고 강조했습니다. 이날 행사에서는 독립유공자 후손 112명에 대한 포상도 진행되었습니다.

### 2. 중동 정세 급변에 따른 긴급 안보·경제 점검
미국과 이스라엘의 이란 공습 및 최고 지도자 사망 소식 등 급박한 중동 상황으로 인해 정부가 긴급 대응에 나섰습니다. 김민석 국무총리는 **'외교·안보 위기대응체계 24시간 가동'**을 지시했으며, 에너지 수급(원유·천연가스) 및 환율·주식시장 등 국내 경제에 미칠 충격을 최소화하기 위한 비상 점검 회의를 주재했습니다.

### 3. 전국적인 3·1절 기념행사와 '폭주족' 집중 단속
천안 독립기념관을 비롯해 전국 각지에서 다양한 3·1운동 재현 행사와 기념 문화행사가 열렸습니다. 국회 의사당 외벽에는 대한민국 임시정부의 법통을 되새기는 '임시의정원 태극기' 대형 현수막이 걸려 눈길을 끌었습니다. 한편, 경찰은 삼일절을 기해 기승을 부리는 이른바 '삼일절 폭주족'을 막기 위해 천안·아산 등 전국 주요 거점에서 대대적인 단속을 벌여 수백 건의 난폭 운전을 적발했습니다.

[소스]
  - yna.co.kr: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGu_lW_7R5HYJ-ApI7Vw0eG0-Nq0rZKrslM1WNEBD_6N6k9OJkCIYSN50l-14ON-msPg_F4GjGrPniNpsa9vveMMTTbAuWLi-QdNvs_GUJ2BuTpBIJ1iTeigmnHq_Kc7rQ

In [39]:
# 9b. 사실 기반 질문
response = client.models.generate_content(
    model=MODEL,
    contents="2024년 노벨 물리학상 수상자는 누구인가요?",
    config=types.GenerateContentConfig(
        tools=[{"google_search": {}}]
    ),
)
print(response.text)

2024년 노벨 물리학상 수상자는 **존 홉필드(John J. Hopfield)**와 **제프리 힌튼(Geoffrey E. Hinton)**입니다.

스웨덴 왕립과학원 노벨 위원회는 2024년 10월 8일, 이 두 사람을 수상자로 발표하며 선정 이유를 다음과 같이 밝혔습니다.

*   **주요 공로:** 인공 신경망을 이용한 머신러닝(기계 학습)의 기초가 되는 발견과 발명
*   **세부 내용:** 
    *   **존 홉필드(미국 프린스턴 대학교 교수):** 물리학적 원리를 바탕으로 정보의 저장과 재구성이 가능한 '홉필드 네트워크'를 개발하여 인공 신경망의 초석을 다졌습니다.
    *   **제프리 힌튼(캐나다 토론토 대학교 교수):** 홉필드 네트워크를 확장하여 데이터 속성을 스스로 찾아내고 이미지를 식별할 수 있는 '볼츠만 머신' 등의 기법을 발명했습니다.

이들의 연구는 오늘날 챗GPT와 같은 생성형 AI와 딥러닝 기술이 발전하는 데 결정적인 토대가 되었으며, 물리학의 도구를 사용하여 현대 인공지능 시대를 열었다는 평가를 받습니다. 인공지능 분야의 연구가 노벨 물리학상을 수상한 것은 이번이 처음이라 과학계에서도 큰 화제가 되었습니다.


---
## 10. URL 컨텍스트
- `tools=[{'url_context': {}}]`

In [40]:
# 10a. 웹페이지 분석
response = client.models.generate_content(
    model=MODEL,
    contents="https://blog.google/technology/ai/ 이 페이지의 최신 글을 요약해줘.",
    config=types.GenerateContentConfig(
        tools=[{"url_context": {}}]
    ),
)
print(response.text)

구글의 AI 블로그 최신 소식에 따르면, 'AI 임팩트 서밋 2026'을 통해 AI 기술의 혜택을 전 세계로 확산하기 위한 글로벌 파트너십과 자금 지원 계획이 발표되었습니다. 또한 전문가 수준의 성능과 빠른 속도를 갖춘 차세대 이미지 생성 및 편집 모델인 '나노 바나나 2(Nano Banana 2)'를 새롭게 공개했습니다. 구글 번역의 문맥 이해 기능 강화와 매사추세츠 AI 교육 이니셔티브 런칭 등 사회 및 교육 분야의 업데이트도 포함되었습니다. 마지막으로 삼성 갤럭시 S26과의 지능형 통합 및 제미나이(Gemini) 앱의 신규 기능을 통해 더욱 고도화된 사용자 경험을 제공하고 있습니다.


In [41]:
# 10b. Google Search + URL Context 결합
response = client.models.generate_content(
    model=MODEL,
    contents="Python 3.13의 새로운 기능을 검색해서 알려줘.",
    config=types.GenerateContentConfig(
        tools=[{"google_search": {}}, {"url_context": {}}]
    ),
)
print(response.text)

Python 3.13은 2024년 10월에 정식 출시되었으며, 성능 향상과 개발자 경험 개선에 초점을 맞춘 중요한 업데이트들을 포함하고 있습니다. 주요 변화를 핵심 카테고리별로 정리해 드립니다.

---

### 1. 프리 스레디드 CPython (GIL 제거, 실험적)
가장 큰 변화는 **글로벌 인터프리터 락(GIL)을 선택적으로 비활성화**할 수 있는 기능입니다.
*   **내용:** 이제 Python을 실행할 때 GIL 없이 실행할 수 있는 빌드 옵션을 제공합니다.
*   **효과:** 다중 코어 CPU에서 여러 스레드를 진정한 병렬로 실행할 수 있게 되어, CPU 집약적인 작업의 성능이 크게 향상될 수 있습니다.
*   **참고:** 아직 실험적 기능이며, 기존 라이브러리와의 호환성 확인을 위해 별도의 설치(Free-threaded build)가 필요합니다.

### 2. 실험적 JIT(Just-In-Time) 컴파일러
Python의 실행 속도를 높이기 위한 **JIT 컴파일러**가 도입되었습니다.
*   **기술:** 'Copy-and-patch' 방식을 사용하여 런타임에 코드의 일부를 기계어로 컴파일합니다.
*   **효과:** 현재는 약 5% 내외의 성능 향상을 보이지만, 향후 Python 성능 최적화의 중요한 기반이 될 기술입니다.

### 3. 개선된 대화형 인터프리터 (REPL)
기본 REPL(터미널 실행창)이 대폭 업그레이드되어 외부 도구(IPython 등) 없이도 편리하게 사용할 수 있습니다.
*   **멀티라인 편집:** 화살표 키로 여러 줄의 코드를 자유롭게 수정할 수 있습니다.
*   **색상 지원:** 프롬프트와 추적(Traceback) 메시지에 색상이 적용되어 가독성이 좋아졌습니다.
*   **도움말 기능:** `F1` 키를 통해 도움말 모드로 바로 진입하거나, 입력 중인 코드에 대한 정보를 쉽게 볼 수 있습니다.
*   **히스토리 브라우징:** 이전에 입력한 명령어를 더 쉽게 검색하고 불러올 수 있습니다.

### 4. 타이핑

---
## 11. 파일 검색 (File Search)
- 업로드된 파일에서 관련 정보 검색

In [42]:
from pathlib import Path

# 테스트 파일 생성
sample_text = """
# 회사 내규 문서

## 제1조 (근무시간)
정규 근무시간은 오전 9시부터 오후 6시까지이다.
점심시간은 오후 12시부터 오후 1시까지이다.

## 제2조 (연차 휴가)
1년 이상 근무한 직원에게는 15일의 연차 휴가가 부여된다.
3년 이상 근무시 매 2년마다 1일이 추가된다.

## 제3조 (재택근무)
주 2회까지 재택근무가 가능하다.
재택근무 신청은 3일 전까지 팀장에게 승인을 받아야 한다.

## 제4조 (경조사 휴가)
본인 결혼: 5일 / 자녀 결혼: 1일 / 배우자 출산: 10일 / 부모 사망: 5일
"""
Path("test_company_rules.txt").write_text(sample_text, encoding="utf-8")
uploaded = client.files.upload(file="test_company_rules.txt")
print(f"업로드 완료: {uploaded.name}")

업로드 완료: files/g1kzfphzvt6i


In [43]:
# 11a. 파일 내용 검색
response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_uri(file_uri=uploaded.uri, mime_type="text/plain"),
        "재택근무 관련 규정을 알려줘.",
    ],
)
print(response.text)

제공해주신 회사 내규 문서의 **제3조(재택근무)**에 따른 규정은 다음과 같습니다.

1.  **횟수:** 주 2회까지 가능합니다.
2.  **신청 기한:** 재택근무를 원하는 날짜의 3일 전까지 신청해야 합니다.
3.  **승인 절차:** 팀장의 승인을 받아야 합니다.


In [45]:
# 11b. 파일 기반 구조화된 응답
class PolicyItem(BaseModel):
    article: str = Field(description="조항 번호")
    title: str = Field(description="조항 제목")
    summary: str = Field(description="핵심 내용 요약")

class PolicySummary(BaseModel):
    policies: List[PolicyItem] = Field(description="정책 목록")

response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_uri(file_uri=uploaded.uri, mime_type="text/plain"),
        "이 문서의 모든 조항을 정리해줘.",
    ],
    config={
        "response_mime_type": "application/json",
        "response_json_schema": PolicySummary.model_json_schema(),
    },
)
result = PolicySummary.model_validate_json(response.text)
for p in result.policies:
    print(f"  {p.article} {p.title}: {p.summary}")

  제1조 근무시간: 정규 근무시간은 09:00~18:00이며 점심시간은 12:00~13:00임.
  제2조 연차 휴가: 1년 이상 근무 시 15일 부여 및 3년 이상 근무 시 2년마다 1일 추가 가산.
  제3조 재택근무: 주 2회 가능하며 3일 전까지 팀장 승인이 필요함.
  제4조 경조사 휴가: 본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 부모 사망 5일 휴가 제공.


---
## 12. 컨텍스트 캐싱
- 대용량 콘텐츠 캐시 → 반복 요청 비용 절감
- NOTE: 최소 토큰 수 요구사항이 있어 충분히 긴 콘텐츠 필요

In [46]:
# 12a. 캐싱 지원 모델 확인
print("캐싱 지원 모델:")
for m in client.models.list():
    if hasattr(m, "supported_actions") and m.supported_actions:
        if "createCachedContent" in m.supported_actions:
            print(f"  {m.name}")

캐싱 지원 모델:
  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-lite-preview-09-2025
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools


In [47]:
# 12b. 캐싱 테스트
long_text = "대한민국의 역사와 문화에 대한 종합 보고서\n\n" + "\n".join(
    [f"""제{i}장: 한국 역사의 {['고대','중세','근세','근대','현대'][i%5]} 시기 - 섹션 {i}
한국의 역사는 수천 년에 걸쳐 발전해 왔습니다. 각 시대마다 고유한 문화와 전통이 형성되었으며,
이는 현재 한국 사회의 기반이 되고 있습니다. 이 시기에는 정치, 경제, 사회, 문화 등 다양한 분야에서
큰 변화가 일어났으며, 교육 제도의 발전과 과학 기술의 진보가 이루어졌습니다.
경제적으로도 농업 기술의 발전과 상업의 확대, 해외 무역의 증가 등이 경제 성장의 주요 동력이 되었습니다.
""" for i in range(1, 101)]
)

Path("cache_test.txt").write_text(long_text, encoding="utf-8")
cache_file = client.files.upload(file="cache_test.txt")
print(f"파일 업로드: {cache_file.name}")

try:
    cache = client.caches.create(
        model=MODEL,
        config=types.CreateCachedContentConfig(
            display_name="korea-history-cache",
            system_instruction="한국 역사 전문가입니다.",
            contents=[cache_file],
            ttl="300s",
        ),
    )
    print(f"캐시 생성: {cache.name}")

    # 질문 1
    r1 = client.models.generate_content(
        model=MODEL,
        contents="이 문서에서 다루는 주요 시대를 나열해줘.",
        config=types.GenerateContentConfig(cached_content=cache.name),
    )
    print(f"\n질문1: {r1.text[:200]}...")
    print(f"토큰: {r1.usage_metadata}")

    # 질문 2 (캐시 재사용)
    r2 = client.models.generate_content(
        model=MODEL,
        contents="경제적 발전에 대해 설명해줘.",
        config=types.GenerateContentConfig(cached_content=cache.name),
    )
    print(f"\n질문2: {r2.text[:200]}...")
    print(f"토큰: {r2.usage_metadata}")

    client.caches.delete(name=cache.name)
    print("\n캐시 삭제 완료")
except Exception as e:
    print(f"캐싱 에러: {e}")

파일 업로드: files/wle9xtfwirwr
캐시 생성: cachedContents/cpg0koe891kofuvvzq6y74yw2n1xuu15crbpmu46

질문1: 제공해주신 보고서의 각 장(Chapter) 제목을 분석한 결과, 이 문서에서 다루고 있는 한국 역사의 주요 시대는 다음과 같이 **5가지**로 분류됩니다.

1.  **고대 시기** (Ancient Period)
2.  **중세 시기** (Medieval Period)
3.  **근세 시기** (Early Modern Period)
4.  **근대 시기**...
토큰: cache_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=12883
)] cached_content_token_count=12883 candidates_token_count=155 candidates_tokens_details=None prompt_token_count=12898 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=12898
)] thoughts_token_count=653 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=13706 traffic_type=None

질문2: 제시해주신 보고서 초안은 각 시대별 제목은 다르지만, 내용은 동일한 문구가 반복되고 있습니다. 한국 역사 전문가로서, 각 시대별(고대-중세-근세-근대-현대) **'경제적 발전'**의 핵심 내용을 체계적으로 정리해 드리겠습니다.

---

### 한국 역사의 시대별 경제 발전 요약

#### 1. 고대 시기 (고조선 ~ 남북국시대) : 농업의 정착과 국가 경...
토큰: cache_tokens_deta

---
## 13. 임베딩 (Embeddings)
- 모델: `gemini-embedding-001` (3+ 버전 미제공)
- 배치 임베딩 / 코사인 유사도

In [48]:
EMBEDDING_MODEL = "gemini-embedding-001"

# 13a. 단일 임베딩
response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="인공지능은 미래 기술의 핵심이다.",
)
emb = response.embeddings[0].values
print(f"[13a] 벡터 차원: {len(emb)}, 앞 5개: {emb[:5]}")

[13a] 벡터 차원: 3072, 앞 5개: [0.00048087392, 0.00512827, -0.0035112284, -0.056956384, -0.0060310545]


In [50]:
import math

# 13b. 배치 임베딩 + 유사도
texts = [
    "인공지능과 머신러닝",
    "딥러닝과 신경망",
    "요리와 레시피",
    "축구와 농구",
    "프로그래밍과 코딩",
]
response = client.models.embed_content(model=EMBEDDING_MODEL, contents=texts)
embeddings = [e.values for e in response.embeddings]

def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb) if na and nb else 0.0

print("[13b] 코사인 유사도:")
pairs = [(0,1), (0,2), (0,4), (2,3)]
for i, j in pairs:
    sim = cosine_sim(embeddings[i], embeddings[j])
    print(f"  '{texts[i]}' vs '{texts[j]}': {sim:.4f}")

[13b] 코사인 유사도:
  '인공지능과 머신러닝' vs '딥러닝과 신경망': 0.7634
  '인공지능과 머신러닝' vs '요리와 레시피': 0.6335
  '인공지능과 머신러닝' vs '프로그래밍과 코딩': 0.6860
  '요리와 레시피' vs '축구와 농구': 0.6019


---
## 14. 토큰 카운팅
- 원격 API / 멀티모달 토큰 카운팅

In [51]:
# 14a. 텍스트 토큰 카운팅
text = "인공지능(AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현한 컴퓨터 과학의 한 분야이다."
r = client.models.count_tokens(model=MODEL, contents=text)
print(f"[14a] '{text[:40]}...' → {r.total_tokens} 토큰")

# 14b. 긴 텍스트
long = "대한민국은 동아시아에 위치한 민주 공화국이다. " * 100
r = client.models.count_tokens(model=MODEL, contents=long)
print(f"[14b] {len(long)}글자 → {r.total_tokens} 토큰")

# 14c. 멀티모달 (이미지 포함)
test_img = Image.new("RGB", (200, 200), "blue")
r_multi = client.models.count_tokens(model=MODEL, contents=[test_img, "설명해줘."])
r_text = client.models.count_tokens(model=MODEL, contents="설명해줘.")
print(f"[14c] 이미지+텍스트: {r_multi.total_tokens} 토큰 (이미지 추정: {r_multi.total_tokens - r_text.total_tokens})")

[14a] '인공지능(AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현...' → 36 토큰
[14b] 2600글자 → 1503 토큰
[14c] 이미지+텍스트: 1095 토큰 (이미지 추정: 1089)


---
## 15. 파일 업로드 (File API)
- 업로드 / 목록 / 삭제

In [54]:
# 15a. 텍스트 파일 업로드
Path("upload_test.txt").write_text("파일 업로드 테스트입니다.\n" * 10, encoding="utf-8")
txt_f = client.files.upload(file="upload_test.txt")
print(f"이름: {txt_f.name}")
print(f"URI: {txt_f.uri}")
print(f"MIME: {txt_f.mime_type}")
print(f"크기: {txt_f.size_bytes} bytes")
print(f"상태: {txt_f.state}")

이름: files/wlbjx0g1ki1u
URI: https://generativelanguage.googleapis.com/v1beta/files/wlbjx0g1ki1u
MIME: text/plain
크기: 370 bytes
상태: FileState.ACTIVE


In [55]:
# 15b. 업로드 파일로 생성
response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_uri(file_uri=txt_f.uri, mime_type="text/plain"),
        "이 파일의 내용을 한줄로 요약해줘.",
    ],
)
print("응답:", response.text)

응답: 제시해주신 내용은 **'파일 업로드 테스트입니다'라는 문구가 반복된 테스트용 문서**입니다.


In [ ]:
# 15c. 파일 목록 + 삭제
print("업로드된 파일 목록:")
for f in client.files.list():
    print(f"  {f.name} ({f.mime_type})")
client.files.delete(name=txt_f.name)
print(f"\n삭제 완료: {txt_f.name}")

업로드된 파일 목록:
  files/etd5tym9upeq (text/plain)

삭제 완료: etd5tym9upeq


---
## 16. 씽킹 모드 (Thinking Mode)
- `thinking_budget` / `include_thoughts`

In [ ]:
# 16a. 씽킹 모드 ON
response = client.models.generate_content(
    model=MODEL,
    contents="한 농부가 강을 건너려 합니다. 늑대, 양, 양배추를 가지고 있고, 배에는 한 번에 하나만 실을 수 있습니다. 늑대와 양을 둘만 남기면 늑대가 양을 먹고, 양과 양배추를 둘만 남기면 양이 양배추를 먹습니다. 어떻게 모두 안전하게 건널 수 있을까요?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_budget=2048, # 생각에 쓸 수 있는 최대 토큰 수
            include_thoughts=True, # 생각 과정을 응답에 포함할지 여부
        )
    ),
)

for part in response.candidates[0].content.parts:
    if hasattr(part, "thought") and part.thought:
        print("[사고 과정]")
        print(part.text[:500] + "..." if len(part.text) > 500 else part.text)
    else:
        print("\n[최종 응답]")
        print(part.text)

print(f"\n토큰 사용: {response.usage_metadata}")

[사고 과정]
**My Thought Process: Solving the Farmer, Wolf, Sheep, and Cabbage Puzzle**

Okay, let's break this down. We've got the classic river crossing problem: farmer, wolf, sheep, and cabbage. The crux is the constraints – the boat only holds the farmer and *one* other item, and the wolf-sheep and sheep-cabbage dynamics present immediate failure scenarios if left unsupervised. My goal is to devise a sequence of moves that safely transports everything.

First trip: I need to get *something* across. Taki...

[최종 응답]
이 문제는 논리적 순서가 중요한 아주 유명한 수수께끼입니다. 핵심은 **두 마리(혹은 물건)만 남겨졌을 때 문제가 생기는 조합을 피하는 것**입니다.

모두 안전하게 건너는 방법은 총 7단계로 다음과 같습니다.

1.  **농부가 양을 데리고 강을 건넙니다.** (강가에는 늑대와 양배추만 남습니다. 늑대는 양배추를 먹지 않으니 안전합니다.)
2.  **농부가 혼자 배를 타고 돌아옵니다.**
3.  **농부가 늑대를 데리고 강을 건넙니다.**
4.  **늑대를 반대편에 내려놓고, 대신 양을 다시 배에 태워 돌아옵니다.** (늑대와 양을 같이 두면 안 되기 때문입니다.)
5.  **양을 원래 자리에 내려놓고, 이번에는 양배추를 배에 실어 강을 건넙니다.** (양과 양배추를 같이 두면 안 되기 때문입니다. 이제 반대편에는 늑대와 양배추가 있게 됩니다.)
6.  **양배추를 늑대 곁에 내려놓고, 농부 혼자 배를 타고 돌아옵니다.**
7.  **마지막으로

In [63]:
# 16b. budget 비교 (0 / 512 / 4096)
for budget in [0, 512, 4096]:
    r = client.models.generate_content(
        model=MODEL,
        contents="15의 계승(factorial)은 얼마인가요?",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_budget=budget,
                include_thoughts=True,
            )
        ),
    )
    thoughts = [p for p in r.candidates[0].content.parts if hasattr(p, "thought") and p.thought]
    answers = [p for p in r.candidates[0].content.parts if not (hasattr(p, "thought") and p.thought) and hasattr(p, "text") and p.text]
    t_len = sum(len(p.text) for p in thoughts)
    a_text = answers[0].text[:80] if answers else "(없음)"
    print(f"  budget={budget:>5}: 사고={t_len:>5}글자, 답변='{a_text}...'")

  budget=    0: 사고=    0글자, 답변='15의 계승(15!, 15 Factorial)은 다음과 같습니다.

**1,307,674,368,000**

(일조 삼천칠십육억 칠천사백삼십육만...'
  budget=  512: 사고= 1329글자, 답변='15의 계승(factorial), 즉 **15!**의 값은 다음과 같습니다.

**1,307,674,368,000**

(읽기: 1조 3,076...'
  budget= 4096: 사고=  945글자, 답변='15의 계승(15!)은 다음과 같습니다.

**1,307,674,368,000**

읽는 법으로는 **1조 3,076억 7,436만 8,000*...'


---
## 17. Live API (실시간 양방향)
- 모델: `gemini-2.0-flash-live-001` (3+ 버전 미제공)
- asyncio 필수

In [2]:
import asyncio

LIVE_MODEL = "gemini-2.0-flash-live-001"

async def test_live_text():
    """17a. Live API 텍스트 세션"""
    async with client.aio.live.connect(
        model=LIVE_MODEL,
        config=types.LiveConnectConfig(
            response_modalities=["TEXT"],
            system_instruction="너는 친절한 한국어 AI 비서야. 짧고 간결하게 답해줘.",
        ),
    ) as session:
        # 메시지 1
        await session.send_client_content(
            turns=types.Content(role="user", parts=[types.Part.from_text("안녕! 너는 누구야?")]),
            turn_complete=True,
        )
        text1 = ""
        async for msg in session.receive():
            if msg.server_content and msg.server_content.model_turn:
                for p in msg.server_content.model_turn.parts:
                    if p.text: text1 += p.text
            if msg.server_content and msg.server_content.turn_complete:
                break
        print(f"응답1: {text1}")

        # 메시지 2
        await session.send_client_content(
            turns=types.Content(role="user", parts=[types.Part.from_text("한국의 수도는?")]),
            turn_complete=True,
        )
        text2 = ""
        async for msg in session.receive():
            if msg.server_content and msg.server_content.model_turn:
                for p in msg.server_content.model_turn.parts:
                    if p.text: text2 += p.text
            if msg.server_content and msg.server_content.turn_complete:
                break
        print(f"응답2: {text2}")

print("=== 17a. Live API 텍스트 세션 ===")
await test_live_text()

=== 17a. Live API 텍스트 세션 ===


APIError: 1008 None. models/gemini-2.0-flash-live-001 is not found for API version v1beta, or is not supported for bidiGenerateContent. Call Lis

In [ ]:
async def test_live_fc():
    """17b. Live API + Function Calling"""
    def get_time(timezone: str) -> str:
        """특정 시간대의 현재 시간을 가져옵니다.

        Args:
            timezone: 시간대 (예: Asia/Seoul)

        Returns:
            현재 시간 문자열
        """
        from datetime import datetime
        import zoneinfo
        try:
            tz = zoneinfo.ZoneInfo(timezone)
            return f"{timezone}: {datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')}"
        except Exception:
            return f"{timezone}: 정보 없음"

    async with client.aio.live.connect(
        model=LIVE_MODEL,
        config=types.LiveConnectConfig(
            response_modalities=["TEXT"],
            tools=[types.Tool(
                function_declarations=[types.FunctionDeclaration.from_callable(callable=get_time)]
            )],
        ),
    ) as session:
        await session.send_client_content(
            turns=types.Content(role="user", parts=[types.Part.from_text("서울의 현재 시간?")]),
            turn_complete=True,
        )
        text = ""
        async for msg in session.receive():
            if msg.tool_call:
                for fc in msg.tool_call.function_calls:
                    print(f"  [함수호출] {fc.name}({fc.args})")
                    result = get_time(**fc.args)
                    print(f"  [결과] {result}")
                    await session.send_tool_response(
                        function_responses=[types.FunctionResponse(name=fc.name, response={"result": result})]
                    )
            if msg.server_content and msg.server_content.model_turn:
                for p in msg.server_content.model_turn.parts:
                    if p.text: text += p.text
            if msg.server_content and msg.server_content.turn_complete:
                break
        print(f"  최종: {text}")

print("=== 17b. Live API + Function Calling ===")
await test_live_fc()

---
## 18. 모델 목록 조회

In [3]:
# 18a. 전체 모델 목록
models = list(client.models.list())
print(f"총 {len(models)}개 모델\n")
for m in models:
    print(f"  {m.name}")

총 44개 모델

  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-exp-image-generation
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-3-1b-it
  models/gemma-3-4b-it
  models/gemma-3-12b-it
  models/gemma-3-27b-it
  models/gemma-3n-e4b-it
  models/gemma-3n-e2b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-2.5-flash-lite-preview-09-2025
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3-pro-image-preview
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/gemini-robotics-er-1.5-preview
  models/gemini-2.5-computer-use-preview-10-2025
  models/deep-res

In [4]:
# 18b. Gemini 3+ 모델 필터
print("Gemini 3+ 모델:")
for m in models:
    if "gemini-3" in m.name:
        info = f"  {m.name}"
        if hasattr(m, "display_name") and m.display_name:
            info += f" ({m.display_name})"
        print(info)

Gemini 3+ 모델:
  models/gemini-3-pro-preview (Gemini 3 Pro Preview)
  models/gemini-3-flash-preview (Gemini 3 Flash Preview)
  models/gemini-3.1-pro-preview (Gemini 3.1 Pro Preview)
  models/gemini-3.1-pro-preview-customtools (Gemini 3.1 Pro Preview Custom Tools)
  models/gemini-3-pro-image-preview (Nano Banana Pro)
  models/gemini-3.1-flash-image-preview (Nano Banana 2)


In [5]:
# 18c. 특정 모델 상세
info = client.models.get(model="gemini-3-flash-preview")
print(f"이름: {info.name}")
if hasattr(info, "display_name"): print(f"표시명: {info.display_name}")
if hasattr(info, "description"): print(f"설명: {info.description}")
if hasattr(info, "input_token_limit"): print(f"입력 한도: {info.input_token_limit}")
if hasattr(info, "output_token_limit"): print(f"출력 한도: {info.output_token_limit}")
if hasattr(info, "supported_actions") and info.supported_actions:
    print(f"지원 액션: {info.supported_actions}")

이름: models/gemini-3-flash-preview
표시명: Gemini 3 Flash Preview
설명: Gemini 3 Flash Preview
입력 한도: 1048576
출력 한도: 65536
지원 액션: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']


---
## 정리

### 기본 기능 (1~18)

| # | 기능 | 모델 |
|---|------|------|
| 1 | 텍스트 생성 / 시스템 인스트럭션 / Safety | `gemini-3-flash-preview` |
| 2 | 멀티턴 채팅 | `gemini-3-flash-preview` |
| 3 | 스트리밍 | `gemini-3-flash-preview` |
| 4 | 멀티모달 입력 (이미지) | `gemini-3-flash-preview` |
| 5 | 이미지 생성 (Gemini) | `gemini-3.1-flash-image-preview` |
| 6 | 코드 실행 | `gemini-3-flash-preview` |
| 7 | 함수 호출 (자동) | `gemini-3-flash-preview` |
| 8 | 구조화된 출력 (JSON/Pydantic) | `gemini-3-flash-preview` |
| 9 | Google 검색 그라운딩 | `gemini-3-flash-preview` |
| 10 | URL 컨텍스트 | `gemini-3-flash-preview` |
| 11 | 파일 검색 | `gemini-3-flash-preview` |
| 12 | 컨텍스트 캐싱 | `gemini-3-flash-preview` |
| 13 | 임베딩 | `gemini-embedding-001` * |
| 14 | 토큰 카운팅 | `gemini-3-flash-preview` |
| 15 | 파일 업로드 | `gemini-3-flash-preview` |
| 16 | 씽킹 모드 | `gemini-3-flash-preview` |
| 17 | Live API | `gemini-2.0-flash-live-001` * |
| 18 | 모델 목록 | - |

### 심화 기능 (19~27)

| # | 기능 | 모델/에이전트 | 난이도 |
|---|------|------|------|
| 19 | Interactions API (에이전트 통합) | `gemini-2.5-flash` | 중급 |
| 20 | Deep Research Agent (자동 리서치) | `deep-research-pro-preview-12-2025` | 중급 |
| 21 | Computer Use (브라우저 조작) | `gemini-2.5-computer-use-preview-10-2025` | 고급 |
| 22 | MCP 연동 (외부 도구 서버) | `gemini-2.5-flash` + MCP 서버 | 고급 |
| 23 | Video Generation (비디오 생성) | `veo-3.1-generate-preview` | 중급 |
| 24 | TTS (텍스트→음성) | `gemini-2.5-flash-preview-tts` | 중급 |
| 25 | Imagen 3 (전용 이미지 생성) | `imagen-3.0-generate-002` | 중급 |
| 26 | Batch API (대량 배치 처리) | 모든 모델 | 중급 |
| 27 | Fine-tuning (커스텀 모델 학습) | `gemini-2.0-flash-lite-001` | 고급 |

\* 전용 모델만 존재 (3+ 버전 없음)

---
## 정리

| # | 기능 | 모델 |
|---|------|------|
| 1 | 텍스트 생성 / 시스템 인스트럭션 / Safety | `gemini-3-flash-preview` |
| 2 | 멀티턴 채팅 | `gemini-3-flash-preview` |
| 3 | 스트리밍 | `gemini-3-flash-preview` |
| 4 | 멀티모달 입력 (이미지) | `gemini-3-flash-preview` |
| 5 | 이미지 생성 | `gemini-3.1-flash-image-preview` |
| 6 | 코드 실행 | `gemini-3-flash-preview` |
| 7 | 함수 호출 (자동) | `gemini-3-flash-preview` |
| 8 | 구조화된 출력 (JSON/Pydantic) | `gemini-3-flash-preview` |
| 9 | Google 검색 그라운딩 | `gemini-3-flash-preview` |
| 10 | URL 컨텍스트 | `gemini-3-flash-preview` |
| 11 | 파일 검색 | `gemini-3-flash-preview` |
| 12 | 컨텍스트 캐싱 | `gemini-3-flash-preview` |
| 13 | 임베딩 | `gemini-embedding-001` * |
| 14 | 토큰 카운팅 | `gemini-3-flash-preview` |
| 15 | 파일 업로드 | `gemini-3-flash-preview` |
| 16 | 씽킹 모드 | `gemini-3-flash-preview` |
| 17 | Live API | `gemini-2.0-flash-live-001` * |
| 18 | 모델 목록 | - |

\* 3+ 전용 모델이 아직 없어 해당 기능 전용 모델 사용

In [6]:
# 19a. 기본 Interaction
interaction = client.interactions.create(
    model="gemini-2.5-flash",
    input="대한민국의 인구는 몇 명인가요? 짧게 답해주세요.",
)
print(f"[19a] ID: {interaction.id}")
print(f"  상태: {interaction.status}")
print(f"  응답: {interaction.outputs[-1].text}")

/var/folders/jt/0q6txlmd2njdnqrqdm7nn7kr0000gn/T/ipykernel_23403/55147371.py:2: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = client.interactions.create(


[19a] ID: v1_ChZ3a3FsYWRpVUJiaVowLWtQNDhxQk9BEhZ3a3FsYWRpVUJiaVowLWtQNDhxQk9B
  상태: completed
  응답: 약 5,167만 명입니다 (2023년 말 통계청 기준).


In [7]:
# 19b. Interaction + 도구 (코드 실행)
interaction = client.interactions.create(
    model="gemini-2.5-flash",
    input="2의 100승을 계산해줘.",
    tools=[{"type": "code_execution"}],
)
print(f"[19b] 코드실행 Interaction:")
print(f"  상태: {interaction.status}")
print(f"  응답: {interaction.outputs[-1].text}")

[19b] 코드실행 Interaction:
  상태: completed
  응답: 2의 100승은 1,267,650,600,228,229,401,496,703,205,376 입니다.


In [8]:
# 19c. Interaction + 함수 호출
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "주어진 도시의 현재 날씨를 가져옵니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름 (예: 서울)"}
        },
        "required": ["city"],
    },
}

interaction = client.interactions.create(
    model="gemini-2.5-flash",
    input="서울 날씨 어때?",
    tools=[weather_tool],
)
print(f"[19c] 함수호출 Interaction:")
print(f"  상태: {interaction.status}")
for output in interaction.outputs:
    if hasattr(output, "text") and output.text:
        print(f"  텍스트: {output.text}")
    if hasattr(output, "function_calls") and output.function_calls:
        for fc in output.function_calls:
            print(f"  함수호출: {fc.name}({fc.args})")

[19c] 함수호출 Interaction:
  상태: requires_action


---
## 20. Deep Research Agent — AI 리서치 에이전트
- `agent='deep-research-pro-preview-12-2025'` 로 복잡한 리서치를 자동 수행
- 백그라운드 실행 (`background=True`) → 폴링으로 결과 확인
- NOTE: 실행 시간이 수 분 소요될 수 있음

In [ ]:
import time

# 20. Deep Research Agent
print("=== 20. Deep Research Agent ===")
print("리서치 시작... (수 분 소요될 수 있음)\n")

initial = client.interactions.create(
    input="2025년에 출시된 주요 AI 모델들을 비교 분석해줘. GPT, Claude, Gemini 중심으로.",
    agent="deep-research-pro-preview-12-2025",
    background=True,
)
print(f"  Interaction ID: {initial.id}")
print(f"  초기 상태: {initial.status}")

# 폴링 (최대 5분)
max_wait = 300
elapsed = 0
while elapsed < max_wait:
    interaction = client.interactions.get(id=initial.id)
    print(f"  [{elapsed}s] 상태: {interaction.status}")

    if interaction.status == "completed":
        print("\n[리서치 완료]")
        report = interaction.outputs[-1].text
        print(report[:1000] + "..." if len(report) > 1000 else report)
        break
    elif interaction.status in ("failed", "cancelled"):
        print(f"\n실패: {interaction.status}")
        break

    time.sleep(15)
    elapsed += 15
else:
    print(f"\n{max_wait}초 타임아웃. 나중에 ID로 다시 조회 가능:")
    print(f"  client.interactions.get(id='{initial.id}')")

---
## 21. Computer Use — 브라우저 자동 조작
- 모델이 스크린샷을 보고 마우스/키보드를 직접 제어
- `gemini-2.5-computer-use-preview-10-2025` 모델 사용
- Playwright 연동 필요 (`pip install playwright`)
- NOTE: 실행하려면 Playwright 브라우저 설치 필요 (`playwright install chromium`)

In [69]:
# 21. Computer Use (브라우저 자동 조작)
# NOTE: 실행 전에 다음 설치 필요:
#   uv add playwright
#   playwright install chromium

COMPUTER_USE_MODEL = "gemini-2.5-computer-use-preview-10-2025"
SCREEN_WIDTH = 1280
SCREEN_HEIGHT = 800

try:
    from playwright.async_api import async_playwright
    from google.genai.types import Content, Part

    async def execute_computer_action(action, page):
        """모델이 요청한 액션을 실행"""
        fc = action.function_call
        name = fc.name
        args = fc.args

        if name == "click":
            x = int(args["x"] * SCREEN_WIDTH)
            y = int(args["y"] * SCREEN_HEIGHT)
            await page.mouse.click(x, y)
            return f"Clicked at ({x}, {y})"
        elif name == "type_text":
            await page.keyboard.type(args["text"])
            return f"Typed: {args['text']}"
        elif name == "scroll":
            await page.mouse.wheel(0, int(args.get("delta_y", 300)))
            return "Scrolled"
        elif name == "navigate":
            await page.goto(args["url"])
            return f"Navigated to {args['url']}"
        return "Unknown action"

    async def run_computer_use():
        pw = await async_playwright().start()
        browser = await pw.chromium.launch(headless=True)
        ctx = await browser.new_context(viewport={"width": SCREEN_WIDTH, "height": SCREEN_HEIGHT})
        page = await ctx.new_page()

        try:
            await page.goto("https://www.google.com")
            print("브라우저 시작: google.com")

            screenshot = await page.screenshot(type="png")
            display(Image.open(io.BytesIO(screenshot)))

            config = types.GenerateContentConfig(
                tools=[types.Tool(computer_use=types.ComputerUse(
                    environment=types.Environment.ENVIRONMENT_BROWSER
                ))],
            )

            contents = [
                Content(role="user", parts=[
                    Part(text="구글 검색창에 'Gemini API'를 검색해줘."),
                    Part.from_bytes(data=screenshot, mime_type="image/png"),
                ])
            ]

            for turn in range(3):
                response = await client.aio.models.generate_content(
                    model=COMPUTER_USE_MODEL,
                    contents=contents,
                    config=config,
                )

                candidate = response.candidates[0]
                contents.append(candidate.content)

                has_actions = any(p.function_call for p in candidate.content.parts)
                if not has_actions:
                    text = " ".join(p.text for p in candidate.content.parts if p.text)
                    print(f"[턴{turn+1}] 에이전트 완료: {text}")
                    break

                for part in candidate.content.parts:
                    if part.function_call:
                        result = await execute_computer_action(part, page)
                        print(f"[턴{turn+1}] 액션: {part.function_call.name} → {result}")

                import asyncio as _aio
                await _aio.sleep(1)
                new_screenshot = await page.screenshot(type="png")
                display(Image.open(io.BytesIO(new_screenshot)))

                contents.append(Content(role="user", parts=[
                    Part(function_response=types.FunctionResponse(
                        name="computer_use", response={"status": "success"}
                    )),
                    Part.from_bytes(data=new_screenshot, mime_type="image/png"),
                ]))
        finally:
            await browser.close()
            await pw.stop()
            print("브라우저 종료")

    await run_computer_use()

except ImportError:
    print("⚠️ Playwright 미설치. 실행하려면:")
    print("  uv add playwright")
    print("  playwright install chromium")
except Exception as e:
    print(f"에러: {e}")

---
## 22. MCP (Model Context Protocol) 연동
- 외부 MCP 서버를 도구로 직접 연결
- `tools=[session]` 으로 MCP 클라이언트 세션을 도구로 전달
- SDK가 자동으로 MCP 도구 발견 → 호출 → 결과 처리
- NOTE: `pip install mcp` 필요

In [70]:
# 22. MCP 연동
# NOTE: 실행 전에 다음 설치 필요:
#   uv add mcp
#   npx -y @philschmid/weather-mcp (Node.js 필요)

try:
    from datetime import datetime
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "@philschmid/weather-mcp"],
        env=None,
    )

    async def run_mcp():
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                prompt = f"What is the weather in Seoul on {datetime.now().strftime('%Y-%m-%d')}?"
                print(f"  프롬프트: {prompt}")

                # MCP session을 tools에 전달 → 자동 도구 호출
                response = await client.aio.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0,
                        tools=[session],  # MCP 세션을 도구로 전달
                    ),
                )
                print(f"  응답: {response.text}")

    print("=== 22. MCP 연동 (날씨 MCP 서버) ===")
    await run_mcp()

except ImportError:
    print("⚠️ MCP 미설치. 실행하려면:")
    print("  uv add mcp")
    print("  Node.js가 설치되어 있어야 합니다 (npx 사용)")
except Exception as e:
    print(f"에러: {e}")

---
## 23. Video Generation (Veo 3.1) — 비디오 생성
- `client.models.generate_videos()` — 텍스트/이미지 → 비디오
- 비동기 long-running operation (생성에 수 분 소요)
- 모델: `veo-3.1-generate-preview`

In [71]:
# 23. Video Generation (Veo 3.1)
VIDEO_MODEL = "veo-3.1-generate-preview"

print("=== 23. 비디오 생성 ===")
print("비디오 생성 시작... (수 분 소요)\n")

try:
    operation = client.models.generate_videos(
        model=VIDEO_MODEL,
        prompt="A serene cherry blossom tree in full bloom, petals gently falling in slow motion with a soft breeze, sunlight filtering through the branches.",
    )
    print(f"  Operation: {operation.name}")

    # 완료 대기
    import time
    max_wait = 300
    elapsed = 0
    while not operation.done and elapsed < max_wait:
        time.sleep(15)
        elapsed += 15
        operation = client.operations.get(operation)
        print(f"  [{elapsed}s] 진행 중...")

    if operation.done:
        print("\n[비디오 생성 완료]")
        if hasattr(operation, "response") and operation.response:
            for i, video in enumerate(operation.response.generated_videos):
                # 비디오 저장
                video_data = video.video
                if hasattr(video_data, "uri") and video_data.uri:
                    print(f"  비디오 {i+1} URI: {video_data.uri}")
                # 바이트로 저장
                if hasattr(video_data, "video_bytes") and video_data.video_bytes:
                    fname = f"23_generated_video_{i}.mp4"
                    with open(fname, "wb") as f:
                        f.write(video_data.video_bytes)
                    print(f"  저장: {fname}")
    else:
        print(f"\n{max_wait}초 타임아웃")

except Exception as e:
    print(f"에러: {e}")
    print("(Veo 모델 접근 권한이 필요할 수 있습니다)")

=== 23. 비디오 생성 ===
비디오 생성 시작... (수 분 소요)

  Operation: models/veo-3.1-generate-preview/operations/rq8ewetw6nhk
  [15s] 진행 중...
  [30s] 진행 중...
  [45s] 진행 중...
  [60s] 진행 중...

[비디오 생성 완료]
  비디오 1 URI: https://generativelanguage.googleapis.com/v1beta/files/0yj8w8vrgn20:download?alt=media


---
## 24. TTS (Text-to-Speech) — 텍스트 → 음성 변환
- Interactions API + `gemini-2.5-flash-preview-tts` 모델
- PCM 오디오 → WAV 파일로 저장
- 다양한 음성(voice) 선택 가능: `kore`, `charon`, `fenrir`, `aoede` 등

In [ ]:
# 24. TTS (Text-to-Speech)
import wave
import base64

TTS_MODEL = "gemini-2.5-flash-preview-tts"

def save_wave(filename, pcm_data, channels=1, rate=24000, sample_width=2):
    """PCM 데이터를 WAV 파일로 저장"""
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm_data)

print("=== 24a. 영어 TTS ===")
try:
    interaction = client.interactions.create(
        model=TTS_MODEL,
        input="Hello! Welcome to the Gemini API Text-to-Speech test. This is amazing!",
        response_modalities=["AUDIO"],
        generation_config={
            "speech_config": {
                "language": "en-us",
                "voice": "kore"
            }
        },
    )

    for output in interaction.outputs:
        if output.type == "audio":
            print(f"  MIME: {output.mime_type}")
            save_wave("24_tts_english.wav", base64.b64decode(output.data))
            print("  저장 완료: 24_tts_english.wav")
except Exception as e:
    print(f"  에러: {e}")

print("\n=== 24b. 한국어 TTS ===")
try:
    interaction = client.interactions.create(
        model=TTS_MODEL,
        input="안녕하세요! Gemini API의 텍스트 음성 변환 테스트입니다. 잘 들리시나요?",
        response_modalities=["AUDIO"],
        generation_config={
            "speech_config": {
                "language": "ko-kr",
                "voice": "kore"
            }
        },
    )

    for output in interaction.outputs:
        if output.type == "audio":
            print(f"  MIME: {output.mime_type}")
            save_wave("24_tts_korean.wav", base64.b64decode(output.data))
            print("  저장 완료: 24_tts_korean.wav")
except Exception as e:
    print(f"  에러: {e}")

---
## 25. Imagen 3 — 전용 이미지 생성 모델
- `imagen-3.0-generate-002` — Gemini 이미지 생성(5번)과 별개의 전용 모델
- 텍스트 → 고품질 이미지 생성
- 5번(Gemini 이미지)과 다른점: 전용 모델이라 이미지 품질에 특화

In [ ]:
# 25. Imagen 3
IMAGEN_MODEL = "imagen-3.0-generate-002"

print("=== 25. Imagen 3 이미지 생성 ===")
try:
    response = client.models.generate_images(
        model=IMAGEN_MODEL,
        prompt="A beautiful Korean traditional hanok house surrounded by autumn maple trees, watercolor painting style",
        config=types.GenerateImagesConfig(
            number_of_images=1,
        ),
    )

    for i, image in enumerate(response.generated_images):
        img_bytes = image.image.image_bytes
        pil_img = Image.open(io.BytesIO(img_bytes))
        display(pil_img)
        pil_img.save(f"25_imagen3_{i}.png")
        print(f"  이미지 {i+1} 저장: 25_imagen3_{i}.png ({pil_img.size})")

except Exception as e:
    print(f"  에러: {e}")
    print("  (Imagen 3 모델 접근 권한이 필요할 수 있습니다)")

---
## 26. Batch API — 대량 요청 배치 처리
- JSONL 파일로 여러 요청을 묶어서 배치 실행
- 비용 50% 절감, 대신 최대 24시간 소요
- `client.batches.create()` → 폴링 → 결과 다운로드

In [ ]:
# 26. Batch API
import json

print("=== 26. Batch API (대량 배치 처리) ===")

# 26a. JSONL 배치 파일 생성
batch_requests = [
    {
        "key": "req-1",
        "request": {
            "contents": [{"parts": [{"text": "한국의 수도는?"}]}],
        },
    },
    {
        "key": "req-2",
        "request": {
            "contents": [{"parts": [{"text": "일본의 수도는?"}]}],
        },
    },
    {
        "key": "req-3",
        "request": {
            "contents": [{"parts": [{"text": "중국의 수도는?"}]}],
        },
    },
]

batch_file = "batch_requests.jsonl"
with open(batch_file, "w") as f:
    for req in batch_requests:
        f.write(json.dumps(req) + "\n")
print(f"  JSONL 생성: {batch_file} ({len(batch_requests)}개 요청)")

# 26b. 파일 업로드
uploaded = client.files.upload(
    file=batch_file,
    config=types.UploadFileConfig(display_name="batch-test", mime_type="jsonl"),
)
print(f"  업로드: {uploaded.name}")

# 26c. 배치 작업 생성
try:
    batch_job = client.batches.create(
        model=MODEL,
        src=uploaded.name,
        config={"display_name": "batch-test-job"},
    )
    print(f"  배치 작업 생성: {batch_job.name}")
    print(f"  상태: {batch_job.state}")

    # 폴링 (배치는 오래 걸릴 수 있으므로 짧게만)
    import time
    completed = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
    max_wait = 120
    elapsed = 0

    while batch_job.state.name not in completed and elapsed < max_wait:
        time.sleep(10)
        elapsed += 10
        batch_job = client.batches.get(name=batch_job.name)
        print(f"  [{elapsed}s] 상태: {batch_job.state.name}")

    if batch_job.state.name == "JOB_STATE_SUCCEEDED":
        print("\n  [배치 완료] 결과 파일:")
        if hasattr(batch_job, "dest") and batch_job.dest:
            print(f"    결과 파일: {batch_job.dest}")
    elif batch_job.state.name in completed:
        print(f"\n  배치 종료: {batch_job.state.name}")
    else:
        print(f"\n  {max_wait}초 대기 후 아직 진행 중. 나중에 확인:")
        print(f"    client.batches.get(name='{batch_job.name}')")

except Exception as e:
    print(f"  에러: {e}")

---
## 27. Fine-tuning — 커스텀 모델 학습
- 자체 데이터로 Gemini 모델을 미세 조정
- `client.tunings.create()` → 학습 → 튜닝된 모델로 추론
- NOTE: 학습에 시간이 소요되며, Vertex AI에서 더 많은 옵션 제공

In [ ]:
# 27. Fine-tuning (커스텀 모델 학습)
# NOTE: 실제 학습은 수십 분~수 시간 소요. 여기서는 생성까지만 테스트.

print("=== 27. Fine-tuning ===")

# 27a. 학습 데이터 준비 (질문-답변 쌍)
training_data = [
    {"text_input": "우리 회사 이름이 뭐야?", "output": "우리 회사는 '스타랩'입니다."},
    {"text_input": "회사 설립일은?", "output": "스타랩은 2020년 3월 15일에 설립되었습니다."},
    {"text_input": "대표이사는 누구야?", "output": "스타랩의 대표이사는 김진섭입니다."},
    {"text_input": "회사 주소가 어디야?", "output": "서울시 강남구 테헤란로 123, 스타랩 빌딩 10층입니다."},
    {"text_input": "주요 사업은?", "output": "AI 기반 SaaS 솔루션 개발이 주요 사업입니다."},
    {"text_input": "직원 수는?", "output": "현재 스타랩에는 약 50명의 직원이 근무하고 있습니다."},
    {"text_input": "회사 비전은?", "output": "AI로 모든 비즈니스를 혁신하는 것이 스타랩의 비전입니다."},
    {"text_input": "고객사가 있어?", "output": "네, 삼성전자, LG전자, 현대자동차 등 대기업 고객사가 있습니다."},
]

print(f"  학습 데이터: {len(training_data)}개 샘플")

# 27b. 튜닝 작업 생성
try:
    tuning_job = client.tunings.create(
        base_model="models/gemini-2.0-flash-lite-001",
        training_dataset=types.TuningDataset(
            inline_data=types.InlineData(examples=training_data),
        ),
        config=types.CreateTuningJobConfig(
            epoch_count=3,
            display_name="starlab-qa-model",
        ),
    )
    print(f"  튜닝 작업 생성: {tuning_job.name}")
    print(f"  상태: {tuning_job.state}")
    print(f"  튜닝 모델명: {tuning_job.tuned_model.model}")

    # 학습 진행 확인 (짧게만)
    import time
    for _ in range(6):  # 최대 1분 대기
        tuning_job = client.tunings.get(name=tuning_job.name)
        print(f"  상태: {tuning_job.state}")
        if tuning_job.state.name in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(10)

    if tuning_job.state.name == "SUCCEEDED":
        # 튜닝된 모델로 추론
        tuned_model = tuning_job.tuned_model.model
        print(f"\n  [학습 완료] 모델: {tuned_model}")
        response = client.models.generate_content(
            model=tuned_model,
            contents="우리 회사 이름이 뭐야?",
        )
        print(f"  응답: {response.text}")
    else:
        print(f"\n  학습이 아직 진행 중입니다. 나중에 확인:")
        print(f"    client.tunings.get(name='{tuning_job.name}')")

except Exception as e:
    print(f"  에러: {e}")
    print("  (Fine-tuning API 접근 권한이 필요할 수 있습니다)")